# DEMARAGE DE LA SESSION SPARK
Kernel Pyspark

In [1]:
# L'exécution de cette cellule démarre l'application Spark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1706016679416_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

<u>Affichage des informations sur la session en cours et liens vers Spark UI</u> :

In [2]:
%%info

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1706016679416_0001,pyspark,idle,Link,Link,None,✔


# IMPORTATION DES LIBRAIRIES
Les packages nécessaires ont été installé via l'étape de **bootstrap** à l'instanciation du serveur.


In [3]:
import pandas as pd
from PIL import Image
import numpy as np
import io
import os
import shutil
import time
import uuid

import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras import Model
from pyspark.sql.functions import col, pandas_udf, PandasUDFType, element_at, split, SparkSession

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# Définition des PATH pour charger les images et enregistrer les résultats

Nous accédons directement à nos **données sur S3** comme si elles étaient **stockées localement**.

In [6]:
PATH = 's3://oc-p8-part4/'
PATH_Data = PATH+'data/Test/'
PATH_Result = PATH+'Results/'
print('PATH:        '+\
      PATH+'\nPATH_Data:   '+\
      PATH_Data+'\nPATH_Result: '+PATH_Result)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

PATH:        s3://oc-p8-part4/
PATH_Data:   s3://oc-p8-part4/data/Test/
PATH_Result: s3://oc-p8-part4/Results/

# PREPROCESSING
## CHARGEMENT DES IMAGES

In [7]:
images = spark.read.format("binaryFile") \
  .option("pathGlobFilter", "*.jpg") \
  .option("recursiveFileLookup", "true") \
  .load(PATH_Data)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [15]:
images.show(10)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+-------------------+------+--------------------+----------+
|                path|   modificationTime|length|             content|     label|
+--------------------+-------------------+------+--------------------+----------+
|s3://oc-p8-part4/...|2024-01-09 10:26:17|  7353|[FF D8 FF E0 00 1...|Watermelon|
|s3://oc-p8-part4/...|2024-01-09 10:26:17|  7350|[FF D8 FF E0 00 1...|Watermelon|
|s3://oc-p8-part4/...|2024-01-09 10:26:17|  7349|[FF D8 FF E0 00 1...|Watermelon|
|s3://oc-p8-part4/...|2024-01-09 10:26:17|  7348|[FF D8 FF E0 00 1...|Watermelon|
|s3://oc-p8-part4/...|2024-01-09 10:26:18|  7328|[FF D8 FF E0 00 1...|Watermelon|
|s3://oc-p8-part4/...|2024-01-09 10:26:18|  7301|[FF D8 FF E0 00 1...|Watermelon|
|s3://oc-p8-part4/...|2024-01-09 10:26:18|  7278|[FF D8 FF E0 00 1...|Watermelon|
|s3://oc-p8-part4/...|2024-01-09 10:26:18|  7275|[FF D8 FF E0 00 1...|Watermelon|
|s3://oc-p8-part4/...|2024-01-09 10:26:18|  7266|[FF D8 FF E0 00 1...|Watermelon|
|s3://oc-p8-part

<u>Je ne conserve que le **path** de l'image et j'ajoute <br />
    une colonne contenant les **labels** de chaque image</u> :

In [9]:
images = images.withColumn('label', element_at(split(images['path'], '/'),-2))
print(images.printSchema())
print(images.select('path','label').show(10,False))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)
 |-- label: string (nullable = true)

None
+---------------------------------------------------+----------+
|path                                               |label     |
+---------------------------------------------------+----------+
|s3://oc-p8-part4/data/Test/Watermelon/r_106_100.jpg|Watermelon|
|s3://oc-p8-part4/data/Test/Watermelon/r_109_100.jpg|Watermelon|
|s3://oc-p8-part4/data/Test/Watermelon/r_108_100.jpg|Watermelon|
|s3://oc-p8-part4/data/Test/Watermelon/r_107_100.jpg|Watermelon|
|s3://oc-p8-part4/data/Test/Watermelon/r_95_100.jpg |Watermelon|
|s3://oc-p8-part4/data/Test/Watermelon/r_79_100.jpg |Watermelon|
|s3://oc-p8-part4/data/Test/Watermelon/r_84_100.jpg |Watermelon|
|s3://oc-p8-part4/data/Test/Watermelon/r_94_100.jpg |Watermelon|
|s3://oc-p8-part4/data/Test/Watermelon/r_88_100.jpg |Watermelon|
|s3://oc-p

## Préparation du modèle 

In [17]:
# Instanciation of MobileNetV2 model, pre trained with ImageNet database
model = MobileNetV2(weights='imagenet',
                    include_top=True,
                    input_shape=(224, 224, 3))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [18]:
# New model without the last layer : feature extraction instead of classification
new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [19]:
# Weights broadcast (Poids du model)
brodcast_weights = sc.broadcast(new_model.get_weights())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [20]:
# VERIFIE l'architecture du model
new_model.summary()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Model: "model_2"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_3 (InputLayer)            [(None, 224, 224, 3) 0                                            
__________________________________________________________________________________________________
Conv1 (Conv2D)                  (None, 112, 112, 32) 864         input_3[0][0]                    
__________________________________________________________________________________________________
bn_Conv1 (BatchNormalization)   (None, 112, 112, 32) 128         Conv1[0][0]                      
__________________________________________________________________________________________________
Conv1_relu (ReLU)               (None, 112, 112, 32) 0           bn_Conv1[0][0]                   
____________________________________________________________________________________________

## CREATION DE FONCTIONS
<u>Mettons cela sous forme de fonction</u> :

In [21]:
def model_fn():
    """
    Returns a MobileNetV2 model with top layer removed 
    and broadcasted pretrained weights.
    """
    model = MobileNetV2(weights='imagenet',
                        include_top=True,
                        input_shape=(224, 224, 3))
    for layer in model.layers:
        layer.trainable = False
    new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)
    new_model.set_weights(brodcast_weights.value)
    return new_model


def preprocess(content):
    """
    Preprocesses raw image bytes for prediction.
    """
    img = Image.open(io.BytesIO(content)).resize([224, 224])
    arr = img_to_array(img)
    return preprocess_input(arr)


def featurize_series(model, content_series):
    """
    Featurize a pd.Series of raw images using the input model.
    :return: a pd.Series of image features
    """
    input = np.stack(content_series.map(preprocess))
    preds = model.predict(input)
    # For some layers, output features will be multi-dimensional tensors.
    # We flatten the feature tensors to vectors for easier storage in Spark DataFrames.
    output = [p.flatten() for p in preds]
    return pd.Series(output)
    return 1


@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    print("test")
    '''
    This method is a Scalar Iterator pandas UDF wrapping our featurization function.
    The decorator specifies that this returns a Spark DataFrame column of type ArrayType(FloatType).

    :param content_series_iter: This argument is an iterator over batches of data, where each batch
                              is a pandas Series of image data.
    '''
    # With Scalar Iterator pandas UDFs, we can load the model once and then re-use it
    # for multiple data batches.  This amortizes the overhead of loading big models.
    model = model_fn()
    for content_series in content_series_iter:
        
        yield featurize_series(model, content_series)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

/mnt1/yarn/usercache/livy/appcache/application_1706016679416_0001/container_1706016679416_0001_01_000001/pyspark.zip/pyspark/sql/pandas/functions.py:392: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

## EXTRACTION DES FEATURES

In [22]:
features_df = images.repartition(24).select(col("path"),
                                            col("label"),
                                            featurize_udf("content").alias("features")
                                           )

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [23]:
print(PATH_Result)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

s3://oc-p8-part4/Results/

In [24]:
# Tant que pas lancer cette commande, il ne se passe rien !
features_df.write.mode("overwrite").parquet(PATH_Result) #42min en local, 5min sur serveur

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## CHARGEMENT DES FEATURES EXTRAITES
<u>On charge les données fraichement enregistrées dans un **DataFrame Spark**</u> :

In [29]:
df_spark = spark.read.parquet(PATH_Result, engine='pyarrow')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# PCA
PySpark library MLlib (on spark df)
> Source:
https://stackoverflow.com/questions/48176849/how-to-get-explained-variance-per-pca-component-in-pyspark

https://spark.apache.org/docs/latest/ml-guide.html

-> best choice:
the RDD-based APIs in the spark.mllib package have entered maintenance mode. The primary Machine Learning API for Spark is now the DataFrame-based API in the spark.ml package.
## IMPORTATION DES LIBRAIRIES

In [33]:
from pyspark.ml.feature import PCA, StandardScaler
from pyspark.sql.functions import udf
from pyspark.ml.linalg import Vectors, VectorUDT

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## PREPROCESSING
We need to convert the features into a dense vector

In [30]:
# Convert the features column to a dense vector
to_vector_udf = udf(lambda arr: Vectors.dense(arr), 
                    returnType=VectorUDT())

# Apply the UDF to create 'dense_features' column
df_spark = df_spark.withColumn("dense_features", 
                               to_vector_udf(df_spark["features"]))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [31]:
df_spark.columns

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['path', 'label', 'features', 'dense_features']

## NORMALISATION 
avec StandardScaler:

In [34]:
# Definie les options de normalisation, et les applique au dataframe Spark
scaler = StandardScaler(
    inputCol = 'dense_features', 
    outputCol = 'scaledFeatures',
    withMean = True,
    withStd = True
).fit(df_spark)

# Execute la transformation/normalisation
df_scaled = scaler.transform(df_spark)
df_scaled.show(6)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+--------------+--------------------+--------------------+--------------------+
|                path|         label|            features|      dense_features|      scaledFeatures|
+--------------------+--------------+--------------------+--------------------+--------------------+
|s3://oc-p8-part4/...|Pineapple Mini|[0.0, 4.063483, 0...|[0.0,4.0634832382...|[-0.7574006047611...|
|s3://oc-p8-part4/...|    Watermelon|[1.0325085, 0.390...|[1.03250849246978...|[1.15588639286388...|
|s3://oc-p8-part4/...|    Watermelon|[0.044492975, 0.0...|[0.04449297487735...|[-0.6749530210954...|
|s3://oc-p8-part4/...| Cucumber Ripe|[1.7044754, 0.015...|[1.70447540283203...|[2.40107281196540...|
|s3://oc-p8-part4/...|     Raspberry|[0.14059186, 0.45...|[0.14059185981750...|[-0.4968772489509...|
|s3://oc-p8-part4/...|   Cauliflower|[0.0, 0.86712885,...|[0.0,0.8671288490...|[-0.7574006047611...|
+--------------------+--------------+--------------------+--------------------+------------

## PCA

In [35]:
n_components = 400 # 600-> 95% de variance, 400-> 92% de variance, 50 -> 60%

# Apply PCA to the scaled features

acp = PCA(
    k = n_components, 
    inputCol = 'scaledFeatures', 
    outputCol = 'pcaFeatures'
).fit(df_scaled)

# Execute la commande SPARk
df_pca = acp.transform(df_scaled).select("path", "label", "pcaFeatures")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [36]:
# Get the explained variance
explained_variance = acp.explainedVariance
total_variance = sum(explained_variance)

# Print the total variance
print(f"Total Variance Explained = {round(total_variance*100,2)} %")

# Print the explained variance for each principal component
for i, ev in enumerate(explained_variance):
    print(f"Principal Component {i + 1}: Explained Variance = {round(ev*100,2)} %")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Total Variance Explained = 95.65 %
Principal Component 1: Explained Variance = 7.67 %
Principal Component 2: Explained Variance = 5.04 %
Principal Component 3: Explained Variance = 4.37 %
Principal Component 4: Explained Variance = 3.02 %
Principal Component 5: Explained Variance = 2.78 %
Principal Component 6: Explained Variance = 2.31 %
Principal Component 7: Explained Variance = 2.22 %
Principal Component 8: Explained Variance = 1.97 %
Principal Component 9: Explained Variance = 1.69 %
Principal Component 10: Explained Variance = 1.6 %
Principal Component 11: Explained Variance = 1.46 %
Principal Component 12: Explained Variance = 1.31 %
Principal Component 13: Explained Variance = 1.28 %
Principal Component 14: Explained Variance = 1.22 %
Principal Component 15: Explained Variance = 1.14 %
Principal Component 16: Explained Variance = 1.09 %
Principal Component 17: Explained Variance = 1.03 %
Principal Component 18: Explained Variance = 1.0 %
Principal Component 19: Explained Varian

## EXPORT LES RESULTATS

In [37]:
# Write the result after PCA to a parquet file
df_pca.write.mode("overwrite").parquet(PATH_Result + "pca_results/")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [41]:
# import 
results = spark.read.parquet(PATH_Result+ "pca_results/", 
                             engine='pyarrow')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [42]:
results.show(6)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+--------------+--------------------+
|                path|         label|         pcaFeatures|
+--------------------+--------------+--------------------+
|s3://oc-p8-part4/...|Pineapple Mini|[-8.6389459409195...|
|s3://oc-p8-part4/...|    Watermelon|[-12.296506064894...|
|s3://oc-p8-part4/...|    Watermelon|[-8.0906314435762...|
|s3://oc-p8-part4/...| Cucumber Ripe|[-6.7765105564183...|
|s3://oc-p8-part4/...|     Raspberry|[-6.5021497913107...|
|s3://oc-p8-part4/...|   Cauliflower|[-5.6660706878106...|
+--------------------+--------------+--------------------+
only showing top 6 rows

#  Suivi de l'avancement des tâches avec le Serveur d'Historique Spark

Il est possible de voir l'avancement des tâches en cours <br />
avec le **serveur d'historique Spark**.

![Accès au serveur d'historique spark](img/EMR_serveur_historique_spark_acces.png)

**Il est également possible de revenir et d'étudier les tâches <br />
qui ont été réalisé, afin de debugger, optimiser les futurs <br />
tâches à réaliser.**

<u>Lorsque la commande "**features_df.write.mode("overwrite").parquet(PATH_Result)**" <br />
était en cours, nous pouvions observer son état d'avancement</u> :

![Progression execution script](img/EMR_jupyterhub_avancement.png)

<u>Le **serveur d'historique Spark** nous permet une vision beaucoup plus précise <br />
de l'exécution des différentes tâche sur les différentes machines du cluster</u> :

![Suivi des tâches spark](img/EMR_SHSpark_01.png)

On peut également constater que notre cluster de calcul a mis <br />
un tout petit peu **moins de 8 minutes** pour traiter les **22 688 images**.

![Temps de traitement](img/EMR_SHSpark_02.png)


In [43]:
# stop the current sparkSession
spark.stop()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…